#  PyTorch Lightning. Разворачивание веб-сервера для использования моделей.

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы: 
* https://lightning.ai/docs/pytorch/stable/starter/introduction.html
* https://lightning.ai/docs/pytorch/stable/levels/core_skills.html
* https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.core.LightningModule.html#lightning.pytorch.core.LightningModule.log
* https://lightning.ai/docs/pytorch/stable/extensions/logging.html
* https://lightning.ai/docs/pytorch/stable/common/progress_bar.html
* https://lightning.ai/docs/pytorch/stable/common/early_stopping.html
* https://lightning.ai/docs/pytorch/1.6.3/api/pytorch_lightning.utilities.model_summary.html#pytorch_lightning.utilities.model_summary.ModelSummary
* https://torchmetrics.readthedocs.io/en/stable/pages/lightning.html
* https://pykit.org/how-to-run-python-flask-app-online-using-ngrok/

## Задачи для совместного разбора

1\. Создайте датасет для регрессии и обучите модель при помощи PyTorch Lightning.

## Задачи для самостоятельного решения

In [1]:
import os
from torch.utils.data import Dataset, random_split, DataLoader
from torchvision import transforms
from PIL import Image
import xml.etree.ElementTree as ET
import random
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from torchvision import models
from torch.nn import functional as F
from torchmetrics.classification import Accuracy

In [18]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

PyTorch version: 2.5.1+rocm6.2
CUDA available: True
CUDA version: None
Number of GPUs: 2
GPU 0: AMD Radeon RX 6700S
GPU 1: AMD Radeon 680M


In [19]:
torch.cuda.set_device(0)
print(f"Using device: {torch.cuda.get_device_name(torch.cuda.current_device())}")

Using device: AMD Radeon RX 6700S


In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


<p class="task" id="1"></p>

1\. Опишите датасет `AnimalDetectionDataset` на основе архива `animals.zip`. Реализуйте `__getitem__` таким образом, чтобы он возвращал три элемента: тензор с изображением, словарь с координатами bounding box и метку объекта. Предусмотрите возможность передавать извне при создании датасета набор преобразований для изображений, преобразование для метки объекта (для кодирования) и флаг, показывающий, нужно ли возвращать исходные или нормированные координаты bounding box.  Разбейте набор данных на обучающее и валидационное множество. При создании датасета не забудьте указать преобразования, соответствующие модели ResNet.

- [ ] Проверено на семинаре

In [5]:
def safe_float(value):
    try:
        return float(value)
    except ValueError:
        print(f"Warning: Некорректное значение {value} не может быть преобразовано в float.")
        return 0.0  # Возвращаем значение по умолчанию, если произошла ошибка

In [6]:
class AnimalDetectionDataset(Dataset):
    def __init__(self, root_dir, transforms=None, label_transforms=None, normalize_bboxes=True):
        """
        :param root_dir: Путь к директории с изображениями и XML файлами.
        :param transforms: Преобразования для изображений (например, для ResNet).
        :param label_transforms: Преобразования для метки (например, кодирование меток).
        :param normalize_bboxes: Если True, нормализуем координаты bounding box, иначе оставляем исходные.
        """
        self.root_dir = root_dir
        self.transforms = transforms
        self.label_transforms = label_transforms
        self.normalize_bboxes = normalize_bboxes

        # Список всех файлов изображений и их соответствующих XML файлов
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]
        self.xml_files = [f.replace('.jpg', '.xml') for f in self.image_files]

        # Выводим пути для проверки
        print(f"Found {len(self.image_files)} image files.")
        print(f"Found {len(self.xml_files)} XML files.")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # Загружаем изображение
        img_name = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_name).convert("RGB")

        # Загружаем XML для извлечения bounding box и метки
        xml_name = os.path.join(self.root_dir, self.xml_files[idx])

        # Проверим, существует ли XML файл
        if not os.path.exists(xml_name):
            print(f"Error: XML файл не найден по пути: {xml_name}")
            return None  # Если XML файл не найден, вернем None или можно обработать по-другому

        try:
            tree = ET.parse(xml_name)
            root = tree.getroot()  # Создаем объект root
        except Exception as e:
            print(f"Error: Не удалось обработать XML файл {xml_name}. Ошибка: {e}")
            return None

        # Извлекаем координаты bounding box с проверкой
        bndbox = root.find('.//bndbox')
        xmin = safe_float(bndbox.find('xmin').text) if bndbox is not None else 0.0
        ymin = safe_float(bndbox.find('ymin').text) if bndbox is not None else 0.0
        xmax = safe_float(bndbox.find('xmax').text) if bndbox is not None else 0.0
        ymax = safe_float(bndbox.find('ymax').text) if bndbox is not None else 0.0

        # Получаем метку объекта (класс)
        label = root.find('.//name').text
        label = 0 if label == 'cat' else 1  # 0 = cat, 1 = dog

        # Преобразуем bounding box (нормализуем или нет)
        bbox = torch.tensor([xmin, ymin, xmax, ymax], dtype=torch.float32)
        if self.normalize_bboxes:
            width, height = image.size
            bbox = bbox / torch.tensor([width, height, width, height], dtype=torch.float32)

        # Применяем преобразования к изображению, если они заданы
        if self.transforms:
            image = self.transforms(image)

        # Преобразуем метку объекта
        if self.label_transforms:
            label = self.label_transforms(label)

        # Возвращаем изображение, bounding box и метку
        return image, {'bbox': bbox, 'label': label}

In [7]:
root_dir = '/home/nsedoff/PycharmProjects/ITiABD-PM22-7-Nikolay-Sedov/DL/Asirra: cat vs dogs/'  # Замените на путь к вашему каталогу

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = AnimalDetectionDataset(root_dir, transforms=transform)

Found 1100 image files.
Found 1100 XML files.


In [8]:
train_size = int(0.8 * len(dataset))  # 80% для обучения
val_size = len(dataset) - train_size  # 20% для валидации
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [9]:
train_size = int(0.8 * len(dataset))  # 80% для обучения
val_size = len(dataset) - train_size  # 20% для валидации
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [10]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

<p class="task" id="2"></p>

2\. Напишите модель для решения задачи выделения объектов в виде объекта `lightning.LightningModule`. Реализуйте двухголовую сеть, одна голова которой предсказывает метку объекта (задача классификации), а вторая голова предсказывает 4 координаты вершин bounding box (задача регрессии). В качестве backbone используйте модель resnet50 из пакета `torchvision`. В качестве функции потерь используйте сумму MSELoss (для расчета ошибки на задаче регрессии) и CrossEntropyLoss (для расчета ошибки на задачи классификации). 

Реализуйте следующий функционал при помощи `lightning` и `torchmetrics`:
* для каждого батча во время обучения рассчитывается значение функции потерь и точности прогнозов, по завершению эпохи метрики усредняются;
* для каждого батча во время валидации рассчитывается значение функции потерь и точности прогнозов, по завершению эпохи метрики усредняются;
* если значение функции потерь не улучшалось в течении 5 эпох, происходит ранняя остановка;
* при создании модель на экран выводится сводка по модели с указанием размерностей выходов слоев;
* для визуализации процесса обучения используется tensorboard.

Используя обученную модель, получите предсказания для изображения кошки и собаки и отрисуйте их. Выполните процедуру, обратную нормализации, чтобы корректно отобразить фотографии.

- [ ] Проверено на семинаре

In [11]:
class ObjectDetectionModel(pl.LightningModule):
    def __init__(self, num_classes=2, lr=1e-4):
        super(ObjectDetectionModel, self).__init__()

        # Загрузка предобученной модели ResNet50
        self.backbone = models.resnet50(pretrained=True)

        # Извлекаем количество входных признаков из последнего слоя ResNet50
        in_features = self.backbone.fc.in_features

        # Заменяем последний слой на Identity, чтобы использовать выходные признаки
        self.backbone.fc = nn.Identity()  # Убираем последний fully connected слой

        # Добавляем головы для классификации и регрессии
        self.classification_head = nn.Linear(in_features, num_classes)
        self.regression_head = nn.Linear(in_features, 4)  # 4 для координат bboxes

        # Для метрики точности
        self.acc_metric = Accuracy(task="multiclass", num_classes=num_classes)

        # Параметры оптимизатора
        self.lr = lr

    def forward(self, x):
        # Пропускаем данные через backbone
        x = self.backbone(x)

        # Классификация
        class_preds = self.classification_head(x)

        # Регрессия для предсказания координат bounding box
        bbox_preds = self.regression_head(x)

        return class_preds, bbox_preds

    def training_step(self, batch, batch_idx):
        images, targets = batch

        # Прогнозы
        class_preds, bbox_preds = self(images)

        # Получаем истинные метки и координаты
        labels = targets['label']
        bboxes = targets['bbox']

        # Расчет потерь
        # Классификация
        classification_loss = F.cross_entropy(class_preds, labels)

        # Регрессия
        bbox_loss = F.mse_loss(bbox_preds, bboxes)

        # Суммарная потеря
        total_loss = classification_loss + bbox_loss

        # Метрики
        acc = self.acc_metric(class_preds.softmax(dim=-1), labels)

        self.log("train_loss", total_loss)
        self.log("train_acc", acc, on_step=True, on_epoch=True)

        return total_loss

    def validation_step(self, batch, batch_idx):
        images, targets = batch

        # Прогнозы
        class_preds, bbox_preds = self(images)

        # Получаем истинные метки и координаты
        labels = targets['label']
        bboxes = targets['bbox']

        # Расчет потерь
        classification_loss = F.cross_entropy(class_preds, labels)
        bbox_loss = F.mse_loss(bbox_preds, bboxes)

        # Суммарная потеря
        total_loss = classification_loss + bbox_loss

        # Метрики
        acc = self.acc_metric(class_preds.softmax(dim=-1), labels)

        self.log("val_loss", total_loss)
        self.log("val_acc", acc, on_step=True, on_epoch=True)

        return total_loss

    def configure_optimizers(self):
        # Используем Adam для оптимизации
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        return optimizer

In [22]:
model = ObjectDetectionModel(num_classes=2, lr=1e-4).to(device)

# Настройка Trainer
trainer = pl.Trainer(
    max_epochs=10,
)

# Загрузка данных (с использованием уже реализованного DataLoader)
trainer.fit(model, train_loader, val_loader)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


RuntimeError: HIP error: invalid device function
HIP kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing AMD_SERIALIZE_KERNEL=3
Compile with `TORCH_USE_HIP_DSA` to enable device-side assertions.


<p class="task" id="3"></p>

3\. Загрузите чекпоинт обученной модели и переведите модель в режим оценки. Допишите функцию `transform_image` и route `predict`. Запустите сервер flask и сделайте POST-запрос к соответствующему эндпоинту. 

При работе в Google Colab вы можете воспользоваться инструментом `ngrok` для проброса локального адреса или запустить сервер Flask в отдельном потоке.

- [ ] Проверено на семинаре

In [ ]:
from PIL import Image
import io
from torchvision.transforms import v2 as T
import torch

def bytes_to_pil(image_bytes: bytes) -> Image:
    return Image.open(io.BytesIO(image_bytes))

def transform_image(image: Image) -> torch.Tensor:
    """Преобразует PIL.Image в тензор"""
    pass

In [ ]:
from flask import Flask
# from flask_ngrok import run_with_ngrok
from flask import Flask, request, jsonify
import threading


ALLOWED_EXTENSIONS = {"png", "jpg", "jpeg"}


def allowed_file(filename):
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS


app = Flask(__name__)  # app name
# run_with_ngrok(app)


@app.route("/predict", methods=["POST"])
def predict():
    if request.method == "POST":
        file = request.files.get("file")
        if file is None or file.filename == "":
            return jsonify({"error": "no file"})
        if not allowed_file(file.filename):
            return jsonify({"error": "format not supported"})
        try:
            img_bytes = file.read()
            image = bytes_to_pil(img_bytes) 
            tensor = transform_image(image)

            # получите прогноз при помощи модели
            data = {
                "bbox": ...,
                "label": ...,
            }
            return jsonify(data)
        except Exception as e:
            return jsonify({"error": f"error during prediction: {e}"})


if __name__ == "__main__":
    threading.Thread(target=lambda: app.run()).start()
    # app.run()

In [ ]:
import requests

resp = requests.post(
    ".../predict",
    files={
        "file": open(
            "path/to/file", "rb"
        )
    },
)

print(resp.text)

## Обратная связь
- [ ] Хочу получить обратную связь по решению